# Регуляризация в машинном обучении (L1, L2, ElasticNet)

### 01. Что такое регуляризация (Regularization) в машинном обучении?
**Регуляризация** — это техника предотвращения переобучения модели путём добавления в функцию потерь (Loss Function) дополнительного слагаемого — **штрафа за сложность модели** (обычно за слишком большие значения весов $w$).

Общая формула функции потерь с регуляризацией:
$$J(w) = \text{Loss}(y, \hat{y}) + \alpha \cdot R(w)$$

где:
* $\text{Loss}(y, \hat{y})$ — исходная функция ошибки (например, MSE или Log-Loss);
* $R(w)$ — регуляризационный член (штраф за величину весов);
* $\alpha$ (или $\lambda$) — коэффициент регуляризации, контролирующий силу штрафа.

---

### 02. Для чего нужна регуляризация? Как она помогает бороться с Overfitting?
* **Зачем нужна:** Чтобы повысить обобщающую способность модели (способность давать точные прогнозы на новых, ранее не виденных данных).
* **Как борется с переобучением (Overfitting):** При переобучении модель слишком сильно подстраивается под шум в обучающей выборке, из-за чего её веса (коэффициенты при признаках) становятся огромными и начинают сильно колебаться при малейшем изменении входных данных. Регуляризация искусственно ограничивает рост весов, делая модель более «гладкой», простой и устойчивой к выбросам.

---

### 03. Что такое L1-регуляризация (Lasso)?
**L1-регуляризация** (в линейной регрессии — **Lasso**, *Least Absolute Shrinkage and Selection Operator*) добавляет к функции потерь **сумму модулей весов**:

$$J(w) = \text{Loss}(y, \hat{y}) + \alpha \sum_{i=1}^{n} |w_i|$$

**Главная особенность:** L1-регуляризация способна **обнулять** веса при малозначимых или шумовых признаках, тем самым выполняя встроенный **отбор признаков (Feature Selection)**.

---

### 04. Что такое L2-регуляризация (Ridge)?
**L2-регуляризация** (в линейной регрессии — **Ridge**, или *гребневая регрессия*) добавляет к функции потерь **сумму квадратов весов**:

$$J(w) = \text{Loss}(y, \hat{y}) + \alpha \sum_{i=1}^{n} w_i^2$$

**Главная особенность:** L2 равномерно сжимает все веса ближе к нулю, но **никогда не обнуляет их полностью**. Это делает модель устойчивой к мультиколлинеарности (сильной корреляции признаков между собой).

---

### 05. Что такое ElasticNet? Как он объединяет L1 и L2?
**ElasticNet** — это гибридный метод регуляризации, который объединяет штрафы L1 и L2 в одной функции потерь:

$$J(w) = \text{Loss}(y, \hat{y}) + \alpha \cdot \rho \sum_{i=1}^{n} |w_i| + \alpha \cdot \frac{1 - \rho}{2} \sum_{i=1}^{n} w_i^2$$

где $\rho \in [0, 1]$ (в `scikit-learn` параметр `l1_ratio`) определяет баланс между L1 и L2:
* при $\rho = 1$ модель превращается в чистый **Lasso**;
* при $\rho = 0$ модель превращается в чистый **Ridge**.

ElasticNet позволяет одновременно отбирать признаки (как L1) и корректно работать с группами сильно скоррелированных признаков (как L2).

---

### 06. Чем отличаются L1, L2 и ElasticNet?

| Критерий | L1 (Lasso) | L2 (Ridge) | ElasticNet |
| :--- | :--- | :--- | :--- |
| **Формула штрафа** | $\alpha \sum \|w_i\|$ | $\alpha \sum w_i^2$ | $\alpha \rho \sum \|w_i\| + \frac{\alpha(1-\rho)}{2} \sum w_i^2$ |
| **Обнуление весов** | Да (создаёт разреженные модели) | Нет (стремятся к нулю, но $\neq 0$) | Да (частично обнуляет) |
| **Отбор признаков (Feature Selection)** | Встроенный | Отсутствует | Встроенный |
| **Мультиколлинеарность** | Плохо (выбирает 1 случайный признак из группы, остальные зануляет) | Отлично (равномерно распределяет веса между коррелирующими признаками) | Хорошо (отбирает или сохраняет группы признаков целиком) |
| **Вычислительная сложность** | Выше (функция не дифференцируема в нуле, нет аналитического решения) | Ниже (гладкая функция, есть точное аналитическое решение) | Средняя |

---

### 07. Что такое коэффициент регуляризации $\alpha$ (или $\lambda$)? Как он влияет на модель?
Коэффициент $\alpha \ge 0$ (в некоторых библиотеках обозначается как $\lambda$, а в `LogisticRegression` из `sklearn` используется обратный параметр $C = \frac{1}{\alpha}$) — это гиперпараметр, задающий **силу штрафа**:

* При $\alpha = 0$ штраф отсутствует, модель сводится к обычной линейной регрессии (OLS).
* Чем больше $\alpha$, тем сильнее сжимаются веса модели и тем сильнее снижается дисперсия (Variance), но растёт смещение (Bias).

---

### 08. Что происходит при слишком сильной и слишком слабой регуляризации?
* **Слишком слабая регуляризация ($\alpha \to 0$):** Модель почти не штрафуется за большие веса $\rightarrow$ запоминает шум в обучающих данных $\rightarrow$ возникает **переобучение (Overfitting)** (высокая точность на `train`, низкая на `test`).
* **Слишком сильная регуляризация ($\alpha \to \infty$):** Веса модели сжимаются почти до нуля, модель становится слишком примитивной (в пределе предсказывает просто константу — среднее значение таргета) $\rightarrow$ возникает **недообучение (Underfitting)** (низкая точность и на `train`, и на `test`).

---

### 09. В каких случаях лучше применять Ridge, Lasso и ElasticNet?
* **Ridge (L2)** — выбор по умолчанию, если:
  * все или большинство признаков полезны для предсказания;
  * в данных присутствует мультиколлинеарность (сильная корреляция между признаками);
  * не требуется автоматический отбор признаков.
* **Lasso (L1)** — лучше применять, если:
  * признаков очень много (десятки или сотни), и предполагается, что лишь малая их часть действительно влияет на целевую переменную;
  * нужна интерпретируемая, разреженная модель (многие веса равны нулю);
  * признаки слабо скоррелированы между собой.
* **ElasticNet** — оптимальный выбор, если:
  * признаков много (часто больше, чем объектов в выборке: $p > n$);
  * среди признаков есть группы сильно скоррелированных переменных, но при этом всё равно нужно отсеять откровенный шум.